![Universidad Espíritu Santo](https://raw.githubusercontent.com/andresrubiop/miar0525-estudiantes/main/utils/logo-uees-color.png)

<div style="background:#821436;color:#FFFFFF;padding:14px 18px;border-radius:10px;margin:6px 0 12px 0"><div style="font-size:12px;letter-spacing:.08em;text-transform:uppercase;opacity:.9">Aprendizaje Automático · MIAR0525 · Semana 4 · Examen práctico integrador · 25 % · entrega hasta el sáb 24/10, 23:59</div><div style="font-size:22px;font-weight:700;margin-top:4px">EXP · Examen práctico integrador · ACS Income</div><div style="font-size:12px;opacity:.9;margin-top:4px">Postgrado · Maestría en Inteligencia Artificial · UEES</div></div>

| | |
|---|---|
| **Caso** | Predecir si el ingreso personal anual supera 50 000 USD con **ACS Income** (Ding et al., 2021; `fetch_acs_income`), auditar el modelo por sexo y raza, explicarlo, documentarlo y planificar su monitoreo. |
| **Resultados de aprendizaje** | RDA1 y RDA2 · integra las cuatro unidades |
| **Tu variante** | Un estado de EE. UU. y una semilla, asignados por `variante.py` a partir de tu código de estudiante |
| **Entregables** | Esta carpeta completa (notebook ejecutado, `src/`, `tests/` en verde, `mlflow.db`, `reports/`) y la presentación o el informe en PDF |
| **Rúbrica** | Reproducibilidad 10 · Datos y pipeline sin fuga 15 · Modelos y selección 20 · MLflow 10 · Equidad 15 · Explicabilidad 10 · Model Card y Datasheet 10 · Monitoreo y riesgos 10 |

**Cómo trabajar.** Abre este notebook desde la carpeta `examen-practico/` del repositorio del curso (en Colab: clona el repositorio y entra a esa carpeta con `%cd`). Las celdas `# TODO` son tuyas; puedes agregar todas las celdas que necesites. Toda afirmación del informe debe salir de una celda de este notebook.

## 0 · Configuración y variante

In [ ]:
import logging
import os
import subprocess
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", message="IProgress not found")
for paquete in ("fairlearn", "shap", "mlflow"):          # en Colab se instalan una sola vez
    try:
        __import__(paquete)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])
import mlflow

logging.getLogger("mlflow").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# raíz del paquete del examen: la carpeta que contiene variante.py
RAIZ = Path(os.environ["MIAR_EXAMEN_RAIZ"]) if os.environ.get("MIAR_EXAMEN_RAIZ") else \
    next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "variante.py").exists()), None)
if RAIZ is None:
    raise RuntimeError("Abre el notebook desde la carpeta examen-practico del repositorio (o entra a ella con %cd en Colab).")
sys.path.insert(0, str(RAIZ))
from src.datos import DICCIONARIO, grupos_sensibles, cargar_datos, particion
from src.pipeline import construir_pipeline
from variante import asignar_variante

UEES = {"vino": "#821436", "azul": "#1F6F8B", "ocre": "#C28E0E", "verde": "#3A7D44", "gris": "#77787B"}
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=list(UEES.values())), "axes.titlecolor": UEES["vino"],
                     "axes.titleweight": "bold", "axes.grid": True, "grid.color": "#EEE8EA", "figure.dpi": 110,
                     "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False})

# TODO: escribe tu código de estudiante; la variante (estado y semilla) sale de él.
CODIGO = "A00000000"
ESTADO, SEMILLA = asignar_variante(CODIGO)
print(f"Variante: estado {ESTADO} · semilla {SEMILLA} · scikit-learn {sklearn.__version__} · MLflow {mlflow.__version__}")

## 1 · Datos y ficha de datos (U1, U4)

Carga tu estado, revisa la calidad y la representación y **completa `reports/datasheet.md`** con las cifras de esta sección.

In [ ]:
X, y = cargar_datos(ESTADO, SEMILLA)
A = grupos_sensibles(X)
print(f"{len(X)} personas · tasa de ingreso > 50 000 USD: {y.mean():.3f} · faltantes: {int(X.isna().sum().sum())}")
print("Categorías distintas:", X[["OCCP", "POBP", "SCHL", "COW"]].nunique().to_dict())
composicion = pd.concat({g: pd.DataFrame({"n": A[g].value_counts(), "proporción": A[g].value_counts(normalize=True).round(3),
                                          "tasa de ingreso alto": y.groupby(A[g]).mean().round(3)}) for g in ("sexo", "raza")})
composicion

**Tu análisis (datos).** ¿Qué grupos tienen pocos casos? ¿Qué tasas base difieren y qué implica eso para las métricas de equidad (M4 §2)? ¿Qué variables podrían funcionar como proxies del sexo o la raza?

## 2 · Partición y pipeline sin fuga (U1)

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

X_tr, X_te, y_tr, y_te = particion(X, y, SEMILLA)          # la prueba se separa primero y se usa una vez
A_tr, A_te = A.loc[X_tr.index], A.loc[X_te.index]
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEMILLA)
base = cross_val_score(construir_pipeline(DummyClassifier(strategy="prior")), X_tr, y_tr, cv=cv, scoring="roc_auc")
print(f"Entrenamiento {len(X_tr)} · prueba {len(X_te)} · AUC del baseline (siempre la tasa base) {base.mean():.3f}")

## 3 · Comparación de modelos con búsqueda y MLflow (U2)

Compara **al menos tres** modelos con validación cruzada solo en el entrenamiento. Registra cada búsqueda en MLflow (`sqlite:///mlflow.db`).

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment(f"EXP-{ESTADO}-{SEMILLA}")
# TODO: define al menos tres modelos con su malla de hiperparámetros (usa construir_pipeline; denso=True para boosting).
candidatos = {
    "logistica": (construir_pipeline(LogisticRegression(max_iter=3000)), {"modelo__C": [0.1, 1, 10]}),
    "modelo_2": (construir_pipeline(...), {}),      # ← completa
    "modelo_3": (construir_pipeline(...), {}),      # ← completa
}
resultados, busquedas = [], {}
for nombre, (pipe, malla) in candidatos.items():
    t0 = time.time()
    with mlflow.start_run(run_name=nombre):
        busq = GridSearchCV(pipe, malla, cv=cv, scoring="roc_auc").fit(X_tr, y_tr)
        i = busq.best_index_
        media, de = busq.cv_results_["mean_test_score"][i], busq.cv_results_["std_test_score"][i]
        mlflow.log_params({k.replace("modelo__", ""): v for k, v in busq.best_params_.items()})
        mlflow.log_metrics({"auc_cv": media, "auc_cv_de": de})
        mlflow.set_tags({"estado": ESTADO, "semilla": str(SEMILLA)})
    busquedas[nombre] = busq
    resultados.append({"modelo": nombre, "AUC CV": round(media, 4), "DE": round(de, 4), "mejores parámetros": busq.best_params_, "segundos": round(time.time() - t0)})
tabla_modelos = pd.DataFrame(resultados).set_index("modelo").sort_values("AUC CV", ascending=False)
tabla_modelos

## 4 · Selección, calibración y umbral (U1, U2)

Elige el modelo y **justifica** (desempeño, diferencia frente a su desviación, costo, explicabilidad). Revisa la calibración con predicciones fuera de muestra y elige el umbral con un criterio de costo definido por ti, **sin usar la prueba**.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import cross_val_predict

# TODO: nombre del modelo elegido y costos de cada error en tu escenario de uso.
ELEGIDO = "..."
COSTO_FP, COSTO_FN = ..., ...
modelo = busquedas[ELEGIDO].best_estimator_
p_oof = cross_val_predict(modelo, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]
frac, media_pred = calibration_curve(y_tr, p_oof, n_bins=10)
umbrales = np.linspace(0.05, 0.95, 91)
costo = [np.mean(np.where((p_oof >= u) & (y_tr == 0), COSTO_FP, 0) + np.where((p_oof < u) & (y_tr == 1), COSTO_FN, 0)) for u in umbrales]
UMBRAL = float(umbrales[int(np.argmin(costo))])
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
axes[0].plot([0, 1], [0, 1], "--", color=UEES["gris"]); axes[0].plot(media_pred, frac, "o-", color=UEES["vino"])
axes[0].set(title=f"Calibración fuera de muestra · Brier {brier_score_loss(y_tr, p_oof):.3f}", xlabel="probabilidad predicha", ylabel="frecuencia observada")
axes[1].plot(umbrales, costo, color=UEES["azul"]); axes[1].axvline(UMBRAL, ls="--", color=UEES["vino"])
axes[1].set(title=f"Costo esperado por persona · umbral {UMBRAL:.2f}", xlabel="umbral", ylabel="costo")
plt.tight_layout()
plt.show()
print(f"Umbral teórico con probabilidades calibradas: {COSTO_FP / (COSTO_FP + COSTO_FN):.2f} · elegido por validación cruzada: {UMBRAL:.2f}")

In [ ]:
modelo.fit(X_tr, y_tr)
p_te = modelo.predict_proba(X_te)[:, 1]
pred_te = (p_te >= UMBRAL).astype(int)
costo_te = np.mean(np.where((pred_te == 1) & (y_te == 0), COSTO_FP, 0) + np.where((pred_te == 0) & (y_te == 1), COSTO_FN, 0))
print(f"Prueba (una sola vez) · AUC {roc_auc_score(y_te, p_te):.3f} · costo por persona {costo_te:.3f} · aprobados {pred_te.mean():.1%}")

## 5 · Auditoría de equidad (U4)

Usa `MetricFrame` por sexo y por raza, con tamaños e intervalos. Elige la métrica de equidad que corresponde a tu escenario y, si hay una brecha relevante, prueba una mitigación y reporta su costo.

In [ ]:
from fairlearn.metrics import MetricFrame, count, false_positive_rate, selection_rate, true_positive_rate
from fairlearn.postprocessing import ThresholdOptimizer
from sklearn.metrics import precision_score

metricas = {"n": count, "aprobados": selection_rate, "TPR": true_positive_rate, "FPR": false_positive_rate,
            "precisión": lambda a, b: precision_score(a, b, zero_division=0)}
auditoria = {}
for g in ("sexo", "raza"):
    auditoria[g] = MetricFrame(metrics=metricas, y_true=y_te, y_pred=pred_te, sensitive_features=A_te[g],
                               n_boot=100, ci_quantiles=[0.025, 0.975], random_state=SEMILLA)
    print(f"— {g} · diferencia máxima: {auditoria[g].difference()[['aprobados', 'TPR', 'FPR', 'precisión']].round(3).to_dict()}")
    display(auditoria[g].by_group.round(3))

In [ ]:
# TODO: mitigación (si corresponde) y su costo; por ejemplo, ThresholdOptimizer con la restricción de tu métrica.
...
tabla_mitigacion

**Tu análisis (equidad).** ¿Qué error perjudica más a una persona en tu escenario? ¿Qué métrica elegiste y por qué? ¿Recomiendas la mitigación, considerando su costo y que usa el atributo sensible al decidir?

## 6 · Explicabilidad (U4)

In [ ]:
import shap
from sklearn.inspection import PartialDependenceDisplay, permutation_importance

sub = X_te.sample(min(5000, len(X_te)), random_state=SEMILLA)
pi = permutation_importance(modelo, sub, y_te.loc[sub.index], scoring="roc_auc", n_repeats=3, random_state=SEMILLA)
importancia = pd.DataFrame({"caída de AUC": pi.importances_mean, "DE": pi.importances_std}, index=sub.columns).sort_values("caída de AUC", ascending=False)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
importancia["caída de AUC"][::-1].plot.barh(ax=axes[0], xerr=importancia["DE"][::-1], color=UEES["vino"])
axes[0].set(title="Importancia por permutación (prueba)", xlabel="caída del AUC")
PartialDependenceDisplay.from_estimator(modelo, sub.sample(300, random_state=SEMILLA), ["AGEP", "WKHP"], kind="both", centered=True,
                                        ax=axes[1:], ice_lines_kw={"alpha": 0.12, "color": UEES["azul"]}, pd_line_kw={"color": UEES["vino"], "lw": 3})
plt.tight_layout()
plt.show()
importancia.round(4).head(5)

In [ ]:
# SHAP local para dos personas: una aprobada y una rechazada (valores en la escala del modelo, sobre variables transformadas)
prep, est = modelo[0], modelo[-1]
Z_ref, Z_te = prep.transform(X_tr.sample(500, random_state=SEMILLA)), prep.transform(X_te)
explicador = shap.LinearExplainer(est, shap.maskers.Independent(Z_ref)) if hasattr(est, "coef_") else shap.TreeExplainer(est)
nombres = prep.get_feature_names_out()
original = pd.Series([n.split("__", 1)[1].rsplit("_", 1)[0] if n.startswith("cat__") else n.split("__", 1)[1] for n in nombres])
casos = [int(np.argmax(pred_te)), int(np.argmin(pred_te))]
for caso in casos:
    ex = explicador(Z_te[[caso]])
    valores = np.asarray(ex.values)[0]
    if valores.ndim == 2:                                   # algunos explicadores devuelven una columna por clase
        valores = valores[:, 1]
    agregados = pd.Series(valores, index=original).groupby(level=0).sum().sort_values(key=abs, ascending=False)
    print(f"Persona {caso} · predicción {p_te[caso]:.3f} · aportes principales: {agregados.head(4).round(3).to_dict()}")

**Tu análisis (explicabilidad).** ¿De qué variables depende el modelo? ¿Hay efectos heterogéneos en las ICE? ¿Qué no se puede concluir de estas explicaciones (M4 §4)?

## 7 · Model Card, plan de monitoreo y registro de riesgos (U4)

Completa `reports/model_card.md`, `reports/plan_monitoreo.md` y `reports/registro_riesgos.md` con las cifras de este notebook. Registra el modelo final en MLflow.

In [ ]:
with mlflow.start_run(run_name=f"final-{ELEGIDO}"):
    mlflow.log_params({"modelo": ELEGIDO, "umbral": UMBRAL, "costo_fp": COSTO_FP, "costo_fn": COSTO_FN})
    mlflow.log_metrics({"auc_prueba": roc_auc_score(y_te, p_te), "costo_prueba": costo_te,
                        **{f"tpr_sexo_{k}": float(v) for k, v in auditoria["sexo"].by_group["TPR"].items()}})
    try:
        mlflow.sklearn.log_model(modelo, name="modelo", input_example=X_te.head(3))
    except Exception:                                       # formato alternativo si skops no acepta algún tipo
        mlflow.sklearn.log_model(modelo, name="modelo", input_example=X_te.head(3), serialization_format="cloudpickle")
print("Corrida final registrada. Explora: mlflow ui --backend-store-uri sqlite:///mlflow.db")

## 8 · Declaración de uso de IA (norma f)

Escribe aquí (y en la sección 9 del informe) qué herramientas de IA generativa o código externo usaste, para qué, y cómo verificaste los resultados. Copia los prompts relevantes en `reports/prompts.md`.

## Autoverificación

In [ ]:
assert len(candidatos) >= 3, "Compara al menos tres modelos."
assert roc_auc_score(y_te, p_te) > base.mean() + 0.2, "El modelo debería superar ampliamente al baseline."
assert 0 < UMBRAL < 1 and set(auditoria) == {"sexo", "raza"}
assert len(mlflow.search_runs(experiment_names=[f"EXP-{ESTADO}-{SEMILLA}"])) >= len(candidatos) + 1, "Registra las búsquedas y el modelo final en MLflow."
print("✓ Estructura del examen completa · recuerda: pytest -q en la carpeta examen-practico y los documentos de reports/")